In [1]:
import numpy as np
import pandas as pd

N_WEEKS = 156
L_MAX = 6

In [2]:
def generate_base_sales(n_weeks=156, base_sales_level=2000, random_state=42):
    np.random.seed(random_state)
    weeks = np.arange(n_weeks)
    seasonal_component = (
        0.15 * np.sin(2 * np.pi * weeks / 52) +
        0.08 * np.sin(4 * np.pi * weeks / 52) +
        0.05 * np.sin(6 * np.pi * weeks / 52)
    )
    quality_factor = np.zeros(n_weeks)
    quality_factor[0] = np.random.normal(0, 0.1)
    for t in range(1, n_weeks):
        quality_factor[t] = (0.95 * quality_factor[t-1] + 0.05 * 0 + np.random.normal(0, 0.02))
    base_price_level = np.zeros(n_weeks)
    base_price_level[0] = 1.0
    for t in range(1, n_weeks):
        base_price_level[t] = (0.9 * base_price_level[t-1] + 0.1 * 1.0 + np.random.normal(0, 0.03))
    market_sentiment = np.zeros(n_weeks)
    market_sentiment[0] = np.random.normal(0, 0.05)
    for t in range(1, n_weeks):
        market_sentiment[t] = 0.7 * market_sentiment[t-1] + np.random.normal(0, 0.03)
    base_sales = (base_sales_level *
                  (1 + seasonal_component) *
                  (1 + quality_factor) *
                  base_price_level ** (-0.9) +
                  market_sentiment * base_sales_level)
    base_sales += np.random.normal(0, base_sales_level * 0.02, n_weeks)
    base_sales = np.maximum(base_sales, base_sales_level * 0.5)
    return pd.DataFrame({'week': weeks, 'base_sales': base_sales,
                         'seasonal_component': seasonal_component,
                         'quality_factor': quality_factor,
                         'base_price_level': base_price_level,
                         'market_sentiment': market_sentiment})

In [10]:
def generate_promotional_calendar(n_weeks=156):
    promotion_schedule = {
        'cyber_week': [46, 47], 'winter_clearance': [2, 3, 4],
        'spring_mid_season': [16], 'summer_clearance': [28, 29],
        'autumn_mid_season': [40],
    }
    promotion_types = ['none'] * n_weeks
    true_discount_rates = np.zeros(n_weeks)
    for year in range(n_weeks // 52 + 1):
        year_offset = year * 52
        for promo_type, weeks in promotion_schedule.items():
            for week in weeks:
                if year_offset + week < n_weeks:
                    promotion_types[year_offset + week] = promo_type
                    if promo_type in ['cyber_week', 'winter_clearance', 'summer_clearance']:
                        true_discount_rates[year_offset + week] = 0.4
                    else:
                        true_discount_rates[year_offset + week] = 0.3
    return promotion_types, true_discount_rates

def calculate_true_promotional_effects(base_sales, base_price_level, promotion_types, true_discount_rates):
    n_weeks = len(base_sales)
    true_promotional_price_level = base_price_level.copy()
    for t in range(n_weeks):
        if promotion_types[t] != 'none':
            true_promotional_price_level[t] = base_price_level[t] * (1 - true_discount_rates[t])
    true_elasticity_by_week = np.zeros(n_weeks)
    base_elasticity = -0.9
    for t in range(n_weeks):
        if promotion_types[t] == 'cyber_week':
            true_elasticity_by_week[t] = base_elasticity * 1.8
        elif promotion_types[t] in ['winter_clearance', 'summer_clearance']:
            true_elasticity_by_week[t] = base_elasticity * 1.5
        elif promotion_types[t] in ['spring_mid_season', 'autumn_mid_season']:
            true_elasticity_by_week[t] = base_elasticity * 1.3
        else:
            true_elasticity_by_week[t] = base_elasticity
    price_ratio = np.maximum(1 - true_discount_rates, 0.5)
    promotional_sales_multipliers = np.power(price_ratio, true_elasticity_by_week)
    promotional_sales_multipliers = np.clip(promotional_sales_multipliers, 0.8, 3.0)
    true_sales_with_promotions = base_sales * promotional_sales_multipliers
    true_promotional_uplift = true_sales_with_promotions - base_sales
    return {'true_promotional_price_level': true_promotional_price_level,
            'true_elasticity_by_week': true_elasticity_by_week,
            'true_sales_with_promotions': true_sales_with_promotions,
            'true_promotional_uplift': true_promotional_uplift}

def generate_measurement_errors(true_promotional_price_level, promotion_types, random_state=42):
    np.random.seed(random_state)
    n_weeks = len(promotion_types)
    observed_promotion_dummy = np.zeros(n_weeks)
    true_promo_indices = [i for i, promo in enumerate(promotion_types) if promo != 'none']
    if len(true_promo_indices) > 0:
        captured_count = int(len(true_promo_indices) * 0.70)
        captured_indices = np.random.choice(true_promo_indices, captured_count, replace=False)
        observed_promotion_dummy[captured_indices] = 1
    non_promo_indices = [i for i, promo in enumerate(promotion_types) if promo == 'none']
    if len(non_promo_indices) > 0:
        false_positive_count = int(len(non_promo_indices) * 0.15)
        false_positive_indices = np.random.choice(non_promo_indices, false_positive_count, replace=False)
        observed_promotion_dummy[false_positive_indices] = 1
    pricing_noise = np.zeros(n_weeks)
    pricing_noise[0] = np.random.normal(0, 0.08)
    for t in range(1, n_weeks):
        pricing_noise[t] = (0.2 * pricing_noise[t-1] + np.random.normal(0, 0.08))
    observed_promotional_price_level = true_promotional_price_level * (1 + pricing_noise)
    observed_promotional_price_level = np.clip(observed_promotional_price_level, 0.5, 1.8)
    return {'observed_promotion_dummy': observed_promotion_dummy,
            'observed_promotional_price_level': observed_promotional_price_level,
            'pricing_noise': pricing_noise}

In [4]:
def generate_quarterly_budget_adjustments(total_sales_series, sensitivity=0.3):
    n_weeks = len(total_sales_series)
    multipliers = np.ones(n_weeks)
    for quarter_start in range(13, n_weeks, 13):
        if quarter_start >= 13:
            prev_quarter_sales = total_sales_series[quarter_start-13:quarter_start].mean()
            baseline_quarter_sales = total_sales_series[max(0, quarter_start-26):quarter_start-13].mean()
            if baseline_quarter_sales > 0:
                performance_ratio = prev_quarter_sales / baseline_quarter_sales
                budget_adjustment = 1 + sensitivity * (performance_ratio - 1)
                budget_adjustment = np.clip(budget_adjustment, 0.8, 1.4)
                end_quarter = min(n_weeks, quarter_start + 13)
                multipliers[quarter_start:end_quarter] = budget_adjustment
    return multipliers

def generate_traditional_marketing_spend(n_weeks, seasonal_component, promotion_types, quarterly_multipliers, random_state=42):
    np.random.seed(random_state)
    channels = {
        'pla': {'base_spend': 60, 'strategic_endogeneity': 0.2},
        'meta': {'base_spend': 40, 'strategic_endogeneity': 0.4},
        'tv': {'base_spend': 10, 'strategic_endogeneity': 0.0}
    }
    spend_data = {}
    for channel, config in channels.items():
        spend = np.zeros(n_weeks)
        base_spend = config['base_spend']
        for t in range(n_weeks):
            log_spend = np.log(base_spend)
            log_spend += np.log(quarterly_multipliers[t])
            log_spend += 0.15 * seasonal_component[t]
            future_promotion_intensity = 0
            for lag in range(1, 4):
                if t + lag < n_weeks and promotion_types[t + lag] != 'none':
                    future_promotion_intensity += 1
            log_spend += config['strategic_endogeneity'] * future_promotion_intensity
            if channel == 'tv':
                week_in_year = t % 52
                if 47 <= week_in_year <= 51:
                    log_spend += np.log(15)
                elif 20 <= week_in_year <= 22:
                    log_spend += np.log(8)
            log_spend += np.random.normal(0, 0.1)
            spend[t] = np.exp(log_spend)
        spend_data[channel] = spend
    return spend_data

def generate_roi_driven_pla_spend(sales_signal, seasonal_component, promotion_types,
                                  quarterly_multiplier, roi_sensitivity=0.8,
                                  roi_response_lag=1, roi_baseline_window=10,
                                  roi_min_multiplier=0.5, roi_max_multiplier=3.0,
                                  random_state=42):
    np.random.seed(random_state)
    n_weeks = len(sales_signal)
    pla_spend = np.zeros(n_weeks)
    base_pla_spend = 60
    sales_signal = np.asarray(sales_signal)
    for t in range(n_weeks):
        log_spend = np.log(base_pla_spend)
        log_spend += np.log(quarterly_multiplier[t])
        log_spend += 0.15 * seasonal_component[t]
        future_promotion_intensity = 0
        for lag in range(1, 4):
            if t + lag < n_weeks and promotion_types[t + lag] != 'none':
                future_promotion_intensity += 1
        log_spend += 0.4 * future_promotion_intensity
        if t >= roi_baseline_window + roi_response_lag:
            recent_start = t - roi_response_lag
            recent_performance = np.mean(sales_signal[recent_start:t])
            baseline_start = max(0, t - roi_baseline_window - roi_response_lag)
            baseline_end = t - roi_response_lag
            baseline_performance = np.mean(sales_signal[baseline_start:baseline_end])
            if baseline_performance > 0:
                performance_ratio = recent_performance / baseline_performance
                roi_multiplier = 1 + roi_sensitivity * (performance_ratio - 1)
                roi_multiplier = np.clip(roi_multiplier, roi_min_multiplier, roi_max_multiplier)
                log_spend += np.log(roi_multiplier)
        log_spend += np.random.normal(0, 0.1)
        pla_spend[t] = np.exp(log_spend)
    return pla_spend

In [5]:
def geometric_adstock(spend_series, adstock_rate, l_max=6):
    effect_weights = np.array([adstock_rate ** i for i in range(l_max)])
    normalized_weights = effect_weights / effect_weights.sum()
    spend_padded = np.concatenate([np.zeros(l_max - 1), spend_series, np.zeros(l_max - 1)])
    adstocked_padded = np.zeros_like(spend_padded, dtype=float)
    for week in range(len(spend_padded)):
        for lag in range(l_max):
            adstocked_padded[week] += spend_padded[week - lag] * normalized_weights[lag]
    return adstocked_padded[(l_max - 1):-(l_max - 1)]

def logistic_saturation(adstocked_series, lam):
    return (1 - np.exp(-lam * adstocked_series)) / (1 + np.exp(-lam * adstocked_series))

CHANNEL_CONFIGS = {
    'pla':  {'adstock_rate': 0.2, 'saturation_lambda': 0.008, 'coefficient': 1100},
    'meta': {'adstock_rate': 0.4, 'saturation_lambda': 0.01,  'coefficient': 600},
    'tv':   {'adstock_rate': 0.7, 'saturation_lambda': 0.006, 'coefficient': 560}
}

def apply_marketing_transformations(spend_data):
    marketing_effects = {}
    total_marketing_uplift = 0
    transformation_details = {}
    for channel, spend in spend_data.items():
        config = CHANNEL_CONFIGS[channel]
        adstocked = geometric_adstock(spend, config['adstock_rate'])
        saturated = logistic_saturation(adstocked, config['saturation_lambda'])
        marketing_effects[channel] = config['coefficient'] * saturated
        total_marketing_uplift += marketing_effects[channel]
        transformation_details[channel] = {
            'avg_spend': spend.mean(), 'avg_adstocked': adstocked.mean(),
            'avg_saturation': saturated.mean(), 'avg_effect': marketing_effects[channel].mean(),
            'implied_roas': marketing_effects[channel].mean() / spend.mean() if spend.mean() > 0 else 0,
            'adstocked': adstocked, 'saturated': saturated}
    return marketing_effects, total_marketing_uplift, transformation_details

In [11]:
def build_dataset(n_weeks=156):
    base_data = generate_base_sales(n_weeks=n_weeks)
    promotion_types, true_discount_rates = generate_promotional_calendar(n_weeks)
    true_promotion_effects = calculate_true_promotional_effects(
        base_data['base_sales'], base_data['base_price_level'], promotion_types, true_discount_rates)
    measured_promotion_effects = generate_measurement_errors(
        true_promotion_effects['true_promotional_price_level'], promotion_types)
    quarterly_multipliers = generate_quarterly_budget_adjustments(
        true_promotion_effects['true_sales_with_promotions'])
    spend_data = generate_traditional_marketing_spend(
        n_weeks, base_data['seasonal_component'], promotion_types, quarterly_multipliers)
    spend_data['pla'] = generate_roi_driven_pla_spend(
        true_promotion_effects['true_sales_with_promotions'].values,
        base_data['seasonal_component'], promotion_types, quarterly_multipliers)
    marketing_effects, total_uplift, details = apply_marketing_transformations(spend_data)
    total_sales = (base_data['base_sales'] + true_promotion_effects['true_promotional_uplift']
                   + sum(marketing_effects.values()))
    dataset = pd.DataFrame({
        'week': range(1, n_weeks + 1),
        'year': [(w-1) // 52 + 1 for w in range(1, n_weeks + 1)],
        'week_of_year': [(w-1) % 52 + 1 for w in range(1, n_weeks + 1)],
        'total_sales': total_sales,
        'base_sales': base_data['base_sales'],
        'promotional_sales_uplift': true_promotion_effects['true_promotional_uplift'],
        'marketing_sales_uplift': sum(marketing_effects.values()),
        'seasonal_component': base_data['seasonal_component'],
        'quality_factor': base_data['quality_factor'],
        'base_price_level': base_data['base_price_level'],
        'market_sentiment': base_data['market_sentiment'],
        'promotion_type': promotion_types,
        'true_discount_rate': true_discount_rates,
        'true_promotion_dummy': [1 if p != 'none' else 0 for p in promotion_types],
        'true_promotional_price_level': true_promotion_effects['true_promotional_price_level'],
        'observed_promotion_dummy': measured_promotion_effects['observed_promotion_dummy'],
        'observed_promotional_price_level': measured_promotion_effects['observed_promotional_price_level'],
        'pla_spend': spend_data['pla'], 'meta_spend': spend_data['meta'], 'tv_spend': spend_data['tv'],
        'pla_effect': marketing_effects['pla'], 'meta_effect': marketing_effects['meta'],
        'tv_effect': marketing_effects['tv'],
        'total_marketing_effect': sum(marketing_effects.values())})
    return dataset, quarterly_multipliers, details

synthetic_dataset, quarterly_multipliers, transformation_details = build_dataset(N_WEEKS)
synthetic_dataset.to_csv('synthetic_mmm_data.csv', index=False)
print(f"Dataset assembled: {len(synthetic_dataset)} weeks, {synthetic_dataset.shape[1]} columns -> synthetic_mmm_data.csv")
synthetic_dataset.head()

Dataset assembled: 156 weeks, 24 columns -> synthetic_mmm_data.csv


,week,year,week_of_year,total_sales,base_sales,promotional_sales_uplift,marketing_sales_uplift,seasonal_component,quality_factor,base_price_level,...,true_promotional_price_level,observed_promotion_dummy,observed_promotional_price_level,pla_spend,meta_spend,tv_spend,pla_effect,meta_effect,tv_effect,total_marketing_effect
0,1,1,1,2896.414275,2239.520824,0.000000,656.893451,0.000000,0.049671,1.000000,...,1.000000,0.0,0.957247,140.332692,107.281573,10.860439,463.268805,187.421404,6.203243,656.893451
1,2,1,2,3196.504325,2203.448705,0.000000,993.055621,0.054956,0.044423,1.055973,...,1.055973,0.0,0.920645,198.097935,140.401496,10.937305,681.164797,301.302236,10.588588,993.055621
2,3,1,3,5495.854098,2320.906782,2304.538094,870.409222,0.106231,0.055155,1.064591,...,0.638755,1.0,0.582051,144.755790,80.293109,11.577548,596.991585,259.394564,14.023073,870.409222
3,4,1,4,5903.182328,2605.964209,2587.585093,709.633027,0.150513,0.082858,1.022393,...,0.613436,1.0,0.639042,106.615036,65.177337,10.249844,475.591387,218.372803,15.668836,709.633027
4,5,1,5,5439.086606,2489.169213,2471.613818,478.303576,0.185183,0.074032,1.039850,...,0.623910,1.0,0.618529,60.262295,37.307287,11.007290,305.694983,155.355628,17.252965,478.303576
